# Commodity Pair Spread Matrix

Calculates the current and historic pairwise difference in price for all commodities in the input universe. A heatmap is displayed showing the latest differential across all security pairs along with the underlying time series data for a chosen security pair.

The results are presented as:
1. A line chart showing the time series of the selected metric and the differential for a chosen security pair.

**Interactive Features**
* Use the input cells/drop-downs to set:
    * The commodities of interest.
    * The metric for the analysis: Price, Volume, Settle, Fair Value, Open Interest
    * The number of lookback days for the analysis.
* Click on a cell in the heatmap to update the time series charts with the underlying data for the relevant security pair.

In [75]:
# Set up your environment
from functools import partial

import numpy as np
import pandas as pd
import ipydatagrid as ipd
import ipywidgets as widgets
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import re

import bql

In [76]:
# Connect to BQL
bq = bql.Service()

In [77]:
def get_data(universe, data_item, data_item_name, lookback_days):
    """Returns a DataFrame containing the time series of the required metric
    and supporting descriptive data."""
    # Add the date range to the relevant data item
    data_item = data_item.with_additional_parameters(
        dates=bq.func.range(f'-{lookback_days}d', '0d'), 
        fill='prev'
    )
    # Required data items
    data_items = {
        data_item_name: data_item,
        'name': bq.data.name(),
        'expiry': bq.data.fut_days_expire(),
    }
    # Build and execute request
    request = bql.Request(universe, data_items)
    response = bq.execute(request)
    # Create the time series DataFrame from the main data item and add
    # the Name and Maturity columns
    df = response.get(data_item_name).df()
    df = df.join(response.get('name').df())
    df = df.join(response.get('expiry').df())
    return df

In [78]:
def calculate_pairwise_diffs(df, data_item_name):
    """Calculates the difference in measure across all possible security
    pairs."""
    # Pivot so that each column is a security and the index is date
    pivot = df.pivot(
        columns='name', 
        index='DATE', 
        values=data_item_name
    )
    # Broadcast subtraction
    diffs = pivot.values[:, None, :] - pivot.values[:, :, None]
    # Reshape to 2D DataFrame with multiindex column headings
    reshape = diffs.reshape(diffs.shape[0], -1)
    cols = pd.MultiIndex.from_product([pivot.columns, pivot.columns])
    diffs = pd.DataFrame(reshape, columns=cols, index=pivot.index)
    return diffs

In [79]:
def sort_by_expiry(df, pivot, axis):
    """Sorts the security names by expiry across either 
    index or column axis."""
    pivot = pivot.reindex(
        df.sort_values('expiry')['name'].unique(), 
        axis=axis
    )
    return pivot

def build_heatmap_df(df, heatmap_df):
    """Stacks the last row of a multi column index DataFrame so that the 
    columns and index are the security names. Sorts the resulting security 
    name columns and index by expiry."""
    heatmap_df = heatmap_df.tail(1).stack().reset_index('DATE', drop=True)
    heatmap_df = sort_by_expiry(df, heatmap_df, axis=0)
    heatmap_df = sort_by_expiry(df, heatmap_df, axis=1)
    return heatmap_df

In [80]:
def get_cell_border(row, col):
    """Obtains the borders of the selected cell for highlighting purposes."""
    return {
        "x0": col - 0.5,
        "y0": row - 0.5,
        "x1": col + 0.5,
        "y1": row + 0.5,
    }

In [81]:
def build_heatmap(df, initial_pair, data_item_name):
    """Constructs a heatmap displaying the pairwise latest metric
    differential for all securities in the received pivoted DataFrame."""
    max_val = abs(df.max().max())

    mask = np.triu(np.ones_like(df, dtype=bool), k=1)  

    masked_data = df.where(mask, other=None) 

    # Create the heatmap
    heatmap = go.FigureWidget(
        go.Heatmap(
            x=df.columns,
            y=df.columns,
            z=masked_data.fillna('').values,
            text=masked_data.fillna('').values,
            zmin=-max_val,
            zmax=max_val,
            texttemplate='%{text:,.0f}' if data_item_name in ['Open Interest', 'Volume'] else '%{text:.2f}',
            hovertemplate='%{y}<br>%{x}<br>%{z:,.0f}<extra></extra>' if data_item_name in ['Open Interest', 'Volume'] else '%{y}<br>%{x}<br>%{z:.2f}<extra></extra>',
            colorscale='RdYlGn',
                    )
                            )
    
    # Set some styling
    heatmap = heatmap.update_layout(
        # Arbitrarily cap height at 1200
        height=min(max(300, len(df) * 20), 1200),
        template='plotly_dark', 
        margin=dict(t=15, l=10, r=10, b=10),
        xaxis=dict(
            tickfont=dict(size=8),
            side='top'
        ),
        yaxis=dict(
            tickfont=dict(size=8), 
        ),
    )
    heatmap.update_yaxes(autorange="reversed")

    # Highlight the initially selected cell
    heatmap.add_shape(
        type="rect",
        **get_cell_border(initial_pair[0], initial_pair[1]),
        line=dict(color="gray", width=3)
    )
    
    return heatmap

In [82]:
# Colors for the two security lines and subplot title
color_x = '#187bcd'
color_y = '#da9100'

line_fig = make_subplots(
    specs=[[{"secondary_y": True}]], 
    subplot_titles=('_')
)

# Add empty traces to the suplobts, which will be filled at run time, or
# when the heatmap is clicked.
# Traces for the time series of the metric for each of the two securities
line_fig.add_trace(
    go.Scatter(
        showlegend=False,
        marker_color=color_x
    )
)

line_fig.add_trace(
    go.Scatter(
        showlegend=False,
        marker_color=color_y
    )
)

# Trace for the time series of the differential of the metric between the 
# two securities
line_fig.add_trace(
    go.Scatter(
        line=dict(color='rgba(0, 0, 0, 0)'), 
        showlegend=False,
        hovertemplate='Differential: %{y:.2f}<extra></extra>',
    ), 
    secondary_y=True,
)
# Add the trace for the over/under line with color shading
line_fig.add_trace(
    go.Scatter(
        fill='tozeroy', 
        fillcolor='rgba(0, 255, 0, 0.3)',
        line=dict(color='rgba(0, 0, 0, 0)'), 
        showlegend=False,
        hoverinfo='skip'
    ), 
    secondary_y=True,
)

line_fig.add_trace(
    go.Scatter(
        fill='tozeroy', 
        fillcolor='rgba(255, 0, 0, 0.3)', 
        line=dict(color='rgba(0, 0, 0, 0)'), 
        showlegend=False,
        hoverinfo='skip'
    ), 
    secondary_y=True,
)

# Set some styling
line_fig = line_fig.update_layout(
    template='plotly_dark', 
    # plot_bgcolor='rgb(33,33,33)',
    # paper_bgcolor='rgb(33,33,33)',
    margin=dict(l=20, r=20, t=30, b=20),
    yaxis2=dict(title='Differential', showgrid=False),
    hovermode="x unified",
    hoverlabel=dict(bgcolor='rgba(88,88,88,0.7)')
)
    
line_fig = go.FigureWidget(line_fig)

In [83]:
def update_subplots(df, diffs, data_item_name, name_x, name_y):
    """Updates the line charts with the time series data for two securities.
    The securities selected will have been identified by the click event on
    a heatmap."""

    # Filter the DataFrame with the time series of the chosen metric to only
    # those rows for the relevant securities
    df_x = df[df['name'] == name_x]
    df_y = df[df['name'] == name_y]
    
    with line_fig.batch_update():
        line_fig.data[0].x = df_x['DATE']
        line_fig.data[0].y = df_x[data_item_name]
        line_fig.data[0].name = name_x
        line_fig.data[0].customdata = df_x['name']
        line_fig.data[0].hovertemplate = ('%{customdata}<br>%{x}<br>'
            + data_item_name 
            + ': %{y:.2f}<br><extra></extra>')

        line_fig.data[1].x = df_y['DATE']
        line_fig.data[1].y = df_y[data_item_name]
        line_fig.data[1].name = name_y
        line_fig.data[1].customdata = df_y['name']
        line_fig.data[1].hovertemplate = ('%{customdata}<br>%{x}<br>'
            + data_item_name 
            + ': %{y:.2f}<br><extra></extra>')
        
        # Update the diff series and the over/under
        diff = diffs[(name_x, name_y)]
        
        line_fig.data[2].x = diff.index
        line_fig.data[2].y = diff.values
        
        line_fig.data[3].x = diff.index
        line_fig.data[3].y = diff.clip(lower=0)
        
        line_fig.data[4].x = diff.index
        line_fig.data[4].y = diff.clip(upper=0)
        
        # Center the yaxis of the series on 0
        diff_max = diff.abs().max()
        line_fig.update_layout(
            yaxis=dict(title=data_item_name),
            yaxis2=dict(
                title=f'{data_item_name} Differential', 
                range=[-diff_max - 0.1, diff_max + 0.1]
            ),
        )
        # Update the first subplot title with the colored security names
        line_fig.layout.annotations[0].update(
            text=f'<span style="color: ' + color_y + '">' 
                + name_y
                + '</span> vs <span style="color: ' + color_x + '">' 
                + name_x 
                + '</span>'
            )

In [84]:
def heatmap_click(df, diffs, data_item_name, heatmap, trace, points, selector):
    """Listener function called when heatmap is clicked."""
    # Identify the clicked securities
    security_a = points.xs[0]
    security_b = points.ys[0]
    # Update the line charts with the data for the clicked securities
    update_subplots(
        df,
        diffs, 
        data_item_name,
        security_a, 
        security_b
    )

    border = get_cell_border(*points.point_inds[0])
    heatmap.update_shapes(**border)

In [85]:
def get_ticker_input():
    """Returns a list of tickers from the input box. Leading and trailing 
    spaces and empty strings are removed."""

    # Add COMB as the source if no source specified
    pattern = re.compile(
        r'^(?P<ticker>.+?)\s+(?!(?:PIT|ELEC|COMB)\b)(?P<sfx>Comdty|Index)\b',
        re.IGNORECASE
    )
    universe = []
    # Split into list of separate tickers, removing any blank entries in
    # the list
    for ticker in ticker_input.value.splitlines():
        universe.append(pattern.sub(r'\g<ticker> COMB \g<sfx>', ticker))
    return universe

# Available commodity base tickers
COMMODITY_OPTIONS = {
    'WTI Crude Oil (CL)': 'CLA Comdty',
    'Brent Crude Oil (CO)': 'COA Comdty',
    'Palladium (PA)': 'PAA Comdty',
    'Gasoil (QS)': 'QSA Comdty',
    'RBOB Gasoline (XB)': 'XBA Comdty',
    'Heating Oil (HO)': 'HOA Comdty',
    'Natural Gas (NG)': 'NGA Comdty',
    'UK Natural Gas (FN)': 'FNA Comdty',
    'Copper COMEX (HG)': 'HGA Comdty',
    'Platinum (PL)': 'PLA Comdty',
    'Gold (GC)': 'GCA Comdty',
    'Shanghai Copper (SCO)': 'SCOA Comdty',
    'Silver (SI)': 'SIA Comdty',
    'Rotterdam Coal (RBT)': 'RBTA Comdty',
    'London Copper (LC)': 'LCA Comdty',
    'Coffee (KC)': 'KCA Comdty',
    'Corn (C)': 'C A Comdty',
    'Cotton (CT)': 'CTA Comdty',
    'Soybeans (S)': 'S A Comdty',
    'Sugar (SB)': 'SBA Comdty',
    'Wheat (W)': 'W A Comdty',
}

def get_start_up_tickers(base_ticker=None):
    """Returns all maturities for the selected commodity base ticker."""
    if base_ticker is None:
        base_ticker = 'CLA Comdty'  # Default to WTI Crude
    request = bql.Request(bq.univ.futures(base_ticker), bq.data.id())
    response = bq.execute(request)
    # Take first 20 maturities
    df = response[0].df().head(20)
    tickers = df['ID()'].to_list()
    return tickers

In [86]:
def run(event=None):
    """Listener function to run code and update output displays when button 
    is clicked."""
    # Clear any remaining outputs
    spinner.layout.visibility = 'visible'
    exception_box.children = []
    
    try:
        # Read the input values
        universe = get_ticker_input()
        data_item_name = metric_picker.value
        data_item = metric_options[data_item_name]
        lookback_days = int(lookback_input.value)

        # Retrieve data from BQL
        df = get_data(universe, data_item, data_item_name, lookback_days)
        diffs = calculate_pairwise_diffs(df, data_item_name)

        df = df.groupby('ID').tail(lookback_days)
        diffs = diffs.tail(lookback_days)

        # Initialize with the first security pair to compare
        initial_pair = [0, 1]

        # Construct the heatmap
        heatmap_df = build_heatmap_df(df, diffs)
        heatmap = build_heatmap(heatmap_df, initial_pair, data_item_name)
        
        # Update the line charts with the first two securities
        update_subplots(
            df,
            diffs, 
            data_item_name, 
            heatmap_df.columns[initial_pair[1]], 
            heatmap_df.columns[initial_pair[0]], 
        )

        # Register the listener function to all traces on the heatmap
        for trace in heatmap.data:
            trace.on_click(
                partial(
                    heatmap_click, 
                    df, 
                    diffs, 
                    data_item_name,
                    heatmap
                )
            )  

        fig_box.children = [heatmap, line_fig]

    except Exception as e:
        exception_box.children = [widgets.Label(f'{e}')]
    
    finally:
        spinner.layout.visibility = 'hidden'

In [87]:
# Create input widgets

metric_options = {
    'Price': bq.data.px_last(),
    'Settle': bq.data.px_settle(),
    'Fair Value': bq.data.fair_value(),
    'Open Interest':bq.data.open_int(),
    'Volume':bq.data.px_volume()
}

# Commodity selector dropdown
commodity_picker = widgets.Dropdown(
    description='Commodity',
    options=list(COMMODITY_OPTIONS.keys()),
    value='WTI Crude Oil (CL)',
    layout={'width': '280px'}
)

def on_commodity_change(change):
    """Update ticker list when commodity selection changes."""
    if change["type"] == "change" and change["name"] == "value":
        base_ticker = COMMODITY_OPTIONS[change["new"]]
        new_tickers = get_start_up_tickers(base_ticker)
        ticker_input.value = '\n'.join(new_tickers)

metric_picker = widgets.Dropdown(
    description='Metric', 
    options=list(metric_options.keys()),
    label='Price',
    layout={'width': '280px'}
)

lookback_input = widgets.Text(
    description='Days', 
    value='90', 
    layout={'width': '280px'}
)

ticker_input = widgets.Textarea(
    description='Ticker List',
    value='\n'.join(get_start_up_tickers(COMMODITY_OPTIONS[commodity_picker.value])),
    placeholder='Add a list of Identifiers - tickers, ISINs, FIGIs etc.',
    rows=10,
    layout={'width': '280px'}
)

# Register the commodity change listener (after ticker_input is defined)
commodity_picker.observe(on_commodity_change)

clear_ticker_input_button = widgets.Button(
    description='Clear Ticker List', 
    layout={'width': '192px', 'margin': '0 0 0 90px'}
)

def clear_tickers(event=None):
    """Clears the ticker input box."""
    ticker_input.value = ''

# Register the clear ticker listener function to the clear ticker button
clear_ticker_input_button.on_click(clear_tickers)

# Widget to allow execution of procedure when clicked
refresh_button = widgets.Button(
    description='Refresh', 
    style={'description_width': 'initial'}, 
    layout={'width': '80px', 'margin': '10px 0 0 0'},
    button_style='success'
)

# Loading widget
spinner = widgets.HTML(
    '''<i class="fa fa-spinner fa-spin" style="font-size: 18px"></i>''',
    layout={'visibility': 'hidden', 'margin': '12px 0 0 10px'}
)

# Empty box to push any exception message into
exception_box = widgets.HBox(layout={'margin': '6px 0 0 -35px'})

fig_box = widgets.VBox()

In [88]:
# Register the run listener function to the go button
refresh_button.on_click(run)

# Collect the display widgets together
ui_display = widgets.VBox(
    [
        widgets.VBox(
            [
                commodity_picker,
                ticker_input, 
                clear_ticker_input_button, 
                metric_picker,
                lookback_input
            ]
        ),
        widgets.HBox([refresh_button, spinner, exception_box]),
        fig_box,
    ]
)

In [89]:
# Run on startup
run()
ui_display

In [90]:
#import nbconvert

# Convert notebook to Python script
#!jupyter nbconvert --to script commodity_pair_spread_matrix_4.ipynb